# TechSolve Ticket Data — Data Quality Audit

**Dataset**: 100,851 tickets x 36 columns (MSP support data)
**Method**: single-column profiling first, then cross-field consistency validation.

> **Every column looks valid in isolation; the problems live between columns.**
> Ten of the thirteen major findings below cannot be caught by any single-column check.

Companion artefacts: `src/m2_clean.py` (treatment pipeline), `outputs/dq_log.csv` (audit trail),
`docs/dq-findings.md` (full bilingual findings).

## 1. Imports & Config

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)


def find_project_root(marker: str = "CLAUDE.md") -> Path:
    p = Path.cwd().resolve()
    for candidate in [p, *p.parents]:
        if (candidate / marker).exists():
            return candidate
    raise FileNotFoundError(f"Could not locate project root (no {marker} found above {p})")


ROOT = find_project_root()
RAW_XLSX = ROOT / "data/raw/TechSolve-Ticket_Data.xlsx"
RAW_CACHE = ROOT / "data/raw/TechSolve-Ticket_Data.parquet"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)

## 2. Load Dataset & Structural Checks

Structure must be verified before any analysis: shape, primary-key uniqueness, id format.
An xlsx of this size reads in ~90s, so we cache to parquet on first load.

In [2]:
if RAW_CACHE.exists():
    df = pd.read_parquet(RAW_CACHE)
else:
    df = pd.read_excel(RAW_XLSX, engine="openpyxl")
    df.to_parquet(RAW_CACHE)

assert df["ticket_id"].is_unique, "ticket_id not unique"
assert df["customer_id"].astype(str).str.match(r"^ACC-\d{5}$").all(), "bad customer_id format"
print(f"shape: {df.shape}")
print(f"ticket_id unique: True | customer_id format valid: True")
print(f"unique customers: {df['customer_id'].nunique():,}")

shape: (100851, 36)
ticket_id unique: True | customer_id format valid: True
unique customers: 5,976


## 3. Column Profiling (single-column pass)

First pass looks at each column alone: distinct values, ranges, distribution shapes.
This catches *surface* issues — inconsistent labels, sentinel dates.

### 3.1 Categorical columns

In [3]:
print(f"category — distinct raw values: {df['category'].str.strip().nunique()}")
vc = df["category"].str.strip().value_counts()
print("\ntop 10 (canonical, ~8k each):"); print(vc.head(10).to_string())
print("\nbottom 10 (variants, <=1k each):"); print(vc.tail(10).to_string())

category — distinct raw values: 32

top 10 (canonical, ~8k each):
category
Feature Request              8187
Payment Problem              8084
Performance Issue            8072
Account Suspension           8024
Subscription Cancellation    8009
Data Sync Issue              7997
Refund Request               7988
Security Concern             7975
Login Issue                  7857
Bug Report                   7826

bottom 10 (variants, <=1k each):
category
Security             987
performance_issue    983
data_sync            963
Payment problem      955
BUG                  788
LOGIN ISSUE          785
Bug report           765
login_issue          755
bug_report           751
Login issue          731


**Finding 1 — category fractured into 32 spellings** (case, snake_case, abbreviations),
affecting 20,831 rows (20.7%). Two-tier frequency structure makes canonical vs variant obvious.
*Treatment: external mapping file -> 10 categories under 5 designed groups; hard assert on unmapped values.*

### 3.2 Numeric ranges

In [5]:
num_cols = ["first_response_time_hours", "resolution_time_hours", "csat_score",
            "issue_complexity_score", "previous_tickets", "monthly_contract_value",
            "customer_tenure_months", "sla_target_hours"]
df[num_cols].describe().round(2).T

,count,mean,std,min,25%,50%,75%,max
first_response_time_hours,100851.0,36.30,20.65,0.5,18.44,36.33,54.19,72.00
resolution_time_hours,100851.0,120.56,68.89,1.0,60.94,120.46,180.18,240.00
csat_score,98834.0,3.00,1.41,1.0,2.00,3.00,4.00,5.00
issue_complexity_score,100851.0,5.52,2.87,1.0,3.00,6.00,8.00,10.00
previous_tickets,100851.0,9.98,6.06,0.0,5.00,10.00,15.00,20.00
monthly_contract_value,100851.0,202.82,235.82,0.0,0.00,109.46,298.67,986.98
customer_tenure_months,100851.0,30.31,17.34,1.0,15.00,30.00,45.00,60.00
sla_target_hours,100851.0,20.97,17.28,4.0,4.00,8.00,24.00,48.00


### 3.3 Date ranges

In [6]:
print("created :", df["ticket_created_date"].min(), "->", df["ticket_created_date"].max())
print("resolved:", df["ticket_resolved_date"].min(), "->", df["ticket_resolved_date"].max())
sent = df["ticket_created_date"].isin(pd.to_datetime(["1970-01-01", "2099-12-31"]))
print(f"sentinel dates: {sent.sum()} rows")
ym = df.loc[~sent, "ticket_created_date"].dt.to_period("M").value_counts().sort_index()
print("\nmonthly volume (first/last 3):")
print(pd.concat([ym.head(3), ym.tail(3)]).to_string())

created : 1970-01-01 00:00:00 -> 2099-12-31 00:00:00
resolved: 1970-01-01 00:00:00 -> 2025-03-18 00:00:00
sentinel dates: 8 rows

monthly volume (first/last 3):
ticket_created_date
2023-07    5630
2023-08    5658
2023-09    5482
2024-11    5449
2024-12    5784
2025-03      10
Freq: M


**Finding 2 — 8 sentinel dates** (five 1970-01-01 epoch values, three 2099-12-31).
**Scope note**: valid data runs Jul 2023 - Dec 2024 (5.4-5.8k/month) with an isolated 10 rows in Mar 2025 —
while the brief requests a "2024-2025" dashboard. *Treatment: flag the 8 sentinels (deleting destroys the record permanently, flagging keeps it available for manual investigation without corrupting metrics that don't touch it. Documents the three new columns (sentinel_date_flag, date_order_invalid, date_needs_review) and notes the downstream interaction with tenure_months_calc.); the range question is a reporting-layer decision — keep all valid history, default the report to 2024 onward, retain 2023 as comparison baseline.*

## 4. Missing Values

Nulls are not one problem. They divide into structural (legitimately absent), business
(process gap), and true missing (not captured) — each with a different treatment.

In [7]:
nulls = df.isna().sum()
print(nulls[nulls > 0].sort_values(ascending=False).to_string())
print("\nResidential rows:", (df["account_type"] == "Residential").sum(),
      " <- compare with industry nulls above")
print("team null == assigned_to null on identical rows:",
      (df["team"].isna() == df["assigned_to"].isna()).all())
print("browser null rate by OS (uniform => not structural):")
print(df.groupby("operating_system")["browser"].apply(lambda s: round(s.isna().mean(), 3)).to_string())

account_manager          66956
billing_contact_email    56363
industry                 31767
company_name             30995
browser                  21905
team                      2017
assigned_to               2017
csat_score                2017
resolution_notes          1205

Residential rows: 31767  <- compare with industry nulls above
team null == assigned_to null on identical rows: True
browser null rate by OS (uniform => not structural):
operating_system
Android    0.217
Linux      0.222
MacOS      0.215
Windows    0.220
iOS        0.212


**Finding 3 — three null classes identified**:
- *Structural*: industry missing on exactly 31,767 rows = the Residential row count to the row. Legitimate; kept null. (This exact alignment is also the evidence that ranks `account_type` authoritative in §5.3.)
- *Business*: team + assigned_to co-missing on the same 2,017 rows -> filled as explicit "Unassigned" bucket. Some of these are already Resolved — a contradiction worth surfacing on the dashboard.
- *True missing*: browser 21.7%, uniform across OS (rules out "mobile has no browser") -> kept null, displayed "Not recorded".

## 5. Cross-field Consistency — the decisive pass

Four test patterns, applied to every eligible column pair. This is where every major
unreliable column was caught.

### 5.1 Two recordings of one fact must agree

If two fields record the same event, they must match. Disagreement means at least one is fabricated.

In [6]:
# (a) resolution duration: stored column vs date difference
diff_h = (df["ticket_resolved_date"] - df["ticket_created_date"]).dt.total_seconds() / 3600
print("corr(date-diff, resolution_time_hours):", round(diff_h.corr(df["resolution_time_hours"]), 4),
      "  <- expected ~1.0")

# (b) SLA breach: provided flag vs recalculation from its own definition
calc = df["resolution_time_hours"] > df["sla_target_hours"]
print("\nflag vs recalculation crosstab (a genuine flag concentrates on the diagonal):")
print(pd.crosstab(calc, df["sla_breached"]).to_string())
print("\nflag rate:", round(df["sla_breached"].eq("Yes").mean(), 3),
      "| recalc rate:", round(calc.mean(), 3))

corr(date-diff, resolution_time_hours): 0.0032   <- expected ~1.0

flag vs recalculation crosstab (a genuine flag concentrates on the diagonal):
sla_breached     No    Yes
row_0                     
False          4169   4193
True          46003  46486

flag rate: 0.503 | recalc rate: 0.917


In [7]:
# (c) exhaustive hypothesis sweep: does ANYTHING predict the flag?
flag = df["sla_breached"].eq("Yes")
hyps = {
    "resolution > target":        df["resolution_time_hours"] > df["sla_target_hours"],
    "first_response > target":    df["first_response_time_hours"] > df["sla_target_hours"],
    "first_response > target/2":  df["first_response_time_hours"] > df["sla_target_hours"] / 2,
    "resolution > 2*target":      df["resolution_time_hours"] > 2 * df["sla_target_hours"],
    "date_diff > target":         diff_h > df["sla_target_hours"],
}
for name, h in hyps.items():
    print(f"  {name:28s} agreement = {(h == flag).mean():.3f}")
print("  (0.5 = unrelated, 1.0 = match)")
spreads = {c: df.groupby(c)["sla_breached"].apply(lambda s: s.eq("Yes").mean()).agg(lambda x: x.max()-x.min())
           for c in ["priority", "status", "team", "category", "region", "escalated"]}
print("\n  flag-rate spread across groups (noise threshold ~0.04):")
for c, s in spreads.items(): print(f"  {c:12s} {s:.3f}")

  resolution > target          agreement = 0.502
  first_response > target      agreement = 0.499
  first_response > target/2    agreement = 0.501
  resolution > 2*target        agreement = 0.503
  date_diff > target           agreement = 0.500
  (0.5 = unrelated, 1.0 = match)

  flag-rate spread across groups (noise threshold ~0.04):
  priority     0.009
  status       0.011
  team         0.013
  category     0.041
  region       0.019
  escalated    0.001


**Finding 4 (most severe) — `sla_breached` is disconnected from the business.** Flag rate 50.3% vs
recalculated 91.8%; agreement ~0.5 against every timing definition; flag rate flat across every dimension.
The flag was generated independently of the fields it should depend on.
**Finding 5 — `ticket_resolved_date` decoupled from `resolution_time`** (corr 0.003).
*Treatment: the true breach history is unrecoverable from a disconnected flag — rebuild, don't repair.
`sla_breached_calc` = resolution_time > target (closed tickets only); original kept as `sla_breached_source`.
`resolution_time_hours` ruled the duration source of truth (it is internally consistent with the
priority/SLA system, validated at 99.2% in §5.4); resolved_date demoted.*

### 5.2 Lifecycle constraints: status must govern field existence

Certain fields only come into existence after ticket closure: resolution date, resolution
duration, satisfaction score, resolution notes. One axiom, four columns tested.

In [4]:
for col in ["ticket_resolved_date", "csat_score", "resolution_notes"]:
    print(f"\nshare of rows WITH {col}, by status:")
    print(df.groupby("status")[col].apply(lambda s: round(s.notna().mean(), 3)).to_string())


share of rows WITH ticket_resolved_date, by status:
status
Closed              1.0
In Progress         1.0
Open                1.0
Pending Customer    1.0
Resolved            1.0

share of rows WITH csat_score, by status:
status
Closed              0.980
In Progress         0.979
Open                0.980
Pending Customer    0.981
Resolved            0.981

share of rows WITH resolution_notes, by status:
status
Closed              0.97
In Progress         1.00
Open                1.00
Pending Customer    1.00
Resolved            0.97


**Finding 6 — the lifecycle axiom is violated by every column it governs**: resolved_date 100%
populated on Open tickets; csat present on 98% of unresolved tickets (~59,400 invalid scores);
resolution_notes *inverted* — 100% on open tickets, 3% missing on closed ones.
*Treatment: `csat_valid` keeps scores only for Resolved/Closed (40,217 rows); notes demoted.
The clean CSAT mean barely moves (3.001 -> 2.996) .*

### 5.3 Per-entity attributes must be stable across rows

In [11]:
nun = df.groupby("customer_id")[["customer_name", "customer_email", "region",
                                 "account_type", "monthly_contract_value"]].nunique()
print("customer_ids with >1 distinct value:")
print((nun > 1).sum().to_string())

print("\naccount_type vs customer_segment (impossible combinations):")
print(pd.crosstab(df["account_type"], df["customer_segment"]).to_string())

print("\ntenure column vs actual account age:")
true_age = (df["ticket_created_date"] - df["account_created_date"]).dt.days / 30.44
print("corr:", round(df["customer_tenure_months"].corr(true_age), 4), " <- expected ~1.0")

customer_ids with >1 distinct value:


customer_name             3594
customer_email            3598
region                       0
account_type                 0
monthly_contract_value       0

account_type vs customer_segment (impossible combinations):
customer_segment  Corporate  Individual  Small Business
account_type                                           
Business              23821       22003           23260
Residential            9953       11706           10108

tenure column vs actual account age:
corr: -0.0037  <- expected ~1.0


**Finding 7 — name/email vary within 60% of accounts while region/type/value never do**:
reclassified as *ticket-level contacts*, not errors — nothing to fix; names excluded from the customer
dimension. Recognising what is NOT an error matters as much as finding what is.
**Finding 8 — `customer_segment` contradicts `account_type`** (9,953 Residential rows labelled
Corporate). account_type is ruled authoritative by the exact null-structure alignment from §4. Segment demoted.
**Finding 9 — `customer_tenure_months` is random** (corr -0.003 with actual account age). Demoted;
recomputed from account_created_date.

### 5.4 Row-wise ordering & business rules

In [12]:
print("first_response > resolution_time:",
      (df["first_response_time_hours"] > df["resolution_time_hours"]).sum(), "rows")
print("account created AFTER its own ticket:",
      (df["account_created_date"] > df["ticket_created_date"]).sum(), "rows")

neg = df[df["ticket_resolved_date"] < df["ticket_created_date"]].copy()
neg["diff_h"] = (neg["ticket_resolved_date"] - neg["ticket_created_date"]).dt.total_seconds() / 3600
print("\nresolved BEFORE created:", len(neg), "rows — violation magnitudes:")
print(neg["diff_h"].value_counts().to_string())

print("\npriority x sla_target (canonical U4/H8/M24/L48):")
print(pd.crosstab(df["priority"], df["sla_target_hours"]).to_string())

print("\nFree/paid contradiction — share of zero monthly value by subscription:")
print(df.groupby("subscription_type")["monthly_contract_value"]
        .apply(lambda s: round((s == 0).mean(), 3)).to_string())

srt = df.sort_values(["customer_id", "ticket_created_date", "ticket_id"])
mono = srt.groupby("customer_id")["previous_tickets"].apply(lambda s: s.is_monotonic_increasing)
print("\nprevious_tickets monotonic within customer:", f"{mono.mean():.1%} of customers")

first_response > resolution_time: 14750 rows
account created AFTER its own ticket: 674 rows

resolved BEFORE created: 53 rows — violation magnitudes:
diff_h
-24.0        50
-669504.0     1
-662808.0     1
-662016.0     1

priority x sla_target (canonical U4/H8/M24/L48):
sla_target_hours     4      8      24     48
priority                                    
High                 63  25047     64     83
Low                  65     67     55  24934
Medium               55     60  25010     77
Urgent            25081     64     73     53

Free/paid contradiction — share of zero monthly value by subscription:
subscription_type
Basic         0.262
Enterprise    0.273
Free          0.326
Premium       0.272



previous_tickets monotonic within customer: 51.7% of customers


In [ ]:
# Certification check for F11/R04: is the 779-row priority/SLA mismatch really random
# data entry noise, or could it be a customer-specific negotiated SLA that R04 would be
# wrong to overwrite? Three independent tests.
canon = {"Urgent": 4, "High": 8, "Medium": 24, "Low": 48}
mismatch = df["sla_target_hours"] != df["priority"].map(canon)

print("distinct customers touched by a mismatch:", df.loc[mismatch, "customer_id"].nunique(),
      "/", df["customer_id"].nunique(), "total")

print("\nmismatch rate by account_type (no clustering => similar rates):")
print(df.groupby("account_type", group_keys=False)
        .apply(lambda g: round((g["sla_target_hours"] != g["priority"].map(canon)).mean(), 4)))

per_cust = df.groupby("customer_id").agg(
    total=("ticket_id", "size"),
    mism=("customer_id", lambda s: mismatch.loc[s.index].sum()),
)
print("\ncorrelation(customer's total ticket count, that customer's mismatch count):",
      round(per_cust["total"].corr(per_cust["mism"]), 4),
      " <- near 1.0 means mismatches scale with volume, i.e. independent per-row noise,")
print("   not a customer-specific override (which would decouple mismatch count from volume)")

print("\ntop-volume mismatched customer, off-diagonal SLA values per priority",
      "(scattered => not one consistent bespoke mapping):")
top_cust = per_cust["mism"].idxmax()
print(df[(df["customer_id"] == top_cust) & mismatch.reindex(df.index, fill_value=False)]
        .groupby(["priority", "sla_target_hours"]).size().to_string())

**Finding 10 — 14,750 rows respond after resolving**  -> flagged, values untouched.
**Finding 11 — 674 accounts created after their own ticket** -> tenure nulled + flagged.
**Finding 12 — 50 of the 53 negative resolutions are exactly -24h**: an exact constant is a
*systematic-shift signature* (date truncation / timezone class), not random error — recoverable (+1 day)
if this were production data. Read the distribution of violations, not just the count.
**Finding 13 — 779 priority/SLA mismatches** against 99.2% conformance.Certified as row-level noise before correcting: mismatches scale linearly with customer ticket volume (r = 0.97) at a constant ~0.8% rate — the heaviest customer (77 mismatches, 11,646 tickets) sits at 0.66%, matching the baseline; no multi-ticket customer sustains an elevated rate, and off-canonical values scatter across all wrong options rather than one alternate mapping. Bespoke SLAs ruled out. → Corrected to canonical (U4/H8/M24/L48), sequenced before the breach recalculation; originals in the DQ log. Statistical evidence, not certainty — production caveat in the Review.
**Finding 14 — monthly_contract_value contradicts subscription_type** (Free 67% charged;
Premium/Enterprise 27% at zero) -> the pair is never used together; revenue analysis deliberately excluded.
**Finding 15 — previous_tickets violates monotonicity for 48% of customers (2,887)**: : A running count can never decrease, so the permitted violation count is zero — one decrease proves the column fabricated. The 52% that pass are mostly trivial (single-ticket customers pass by default; two-ticket customers pass by coin flip). Treatment: demoted; rebuilt as previous_tickets_calc, a time-ordered cumulative count per customer.

## 6. Duplicate Check

With the primary key included, duplicates are always zero — so the key is excluded first.
This check runs on raw columns and must precede derived-column creation in the pipeline
(cumulative counts break exact equality within duplicate pairs).

In [13]:
dup_mask = df.drop(columns=["ticket_id"]).duplicated(keep=False)
print("rows in duplicate groups:", dup_mask.sum(),
      "| flagged as duplicates (keep='first'):", df.drop(columns=["ticket_id"]).duplicated().sum())
cols = ["ticket_id", "customer_id", "category", "ticket_created_date", "resolution_time_hours"]
print("\nexample pair:")
print(df[dup_mask].sort_values(["customer_id", "ticket_created_date"])[cols].head(2).to_string(index=False))

rows in duplicate groups: 86 | flagged as duplicates (keep='first'): 43

example pair:
 ticket_id customer_id          category ticket_created_date  resolution_time_hours
      2882   ACC-00135 Performance Issue          2023-07-16                 230.77
      2897   ACC-00135 Performance Issue          2023-07-16                 230.77


**Finding 16 — 43 near-duplicate tickets** (identical on all 35 non-key columns, durations matching
to two decimals). *Treatment: flagged `is_duplicate` (keep='first'), never deleted — volume metrics filter
them out while the duplicate-submission rate itself stays available as an operational signal.*

## 7. Text Field Independence

Three text-bearing fields (description, resolution notes, category) should be strongly related.
Test: for each text value, the share of its most common counterpart (~1.0 = linked, ~0.10 = random).

In [14]:
def top_share(a, b):
    ct = pd.crosstab(df[a].str.strip().str[:40], df[b].str.strip())
    return (ct.max(axis=1) / ct.sum(axis=1)).describe().round(3)[["min", "max"]]

print("description templates:", df["issue_description"].nunique(),
      "| note templates:", df["resolution_notes"].nunique())
print("\ndescription vs category  ", dict(top_share("issue_description", "category")))
print("notes       vs category  ", dict(top_share("resolution_notes", "category")))
print("notes       vs description", dict(top_share("resolution_notes", "issue_description")))

description templates: 10 | note templates: 10

description vs category   {'min': np.float64(0.081), 'max': np.float64(0.086)}
notes       vs category   {'min': np.float64(0.082), 'max': np.float64(0.09)}
notes       vs description {'min': np.float64(0.103), 'max': np.float64(0.107)}


**Finding 17 — the three text fields are mutually independent** (top-counterpart shares 0.08-0.11
across all pairs). A refund-request description sits under Bug Report with credential-reset notes.
This is fabricated text, not dirty text — the information never existed, so the fix is *containment,
not repair*.It also vetoes deriving sub-categories from text — hence the
designed hierarchy category_group(5) -> category(10).

## 8. Findings Summary & Export

Structured findings table — the hand-off artefact to the cleaning pipeline (`m2_clean.py`)
and the source for the Review write-up.

In [15]:
findings = pd.DataFrame([
 ("F01","HIGH","sla_breached flag disconnected from handling time (50.3% vs 91.8% recalc; no predictor found)","crosstab + exhaustive hypothesis sweep","rebuild sla_breached_calc (closed only); keep source column"),
 ("F02","HIGH","ticket_resolved_date decoupled from resolution_time (corr 0.003; populated on open tickets; 53 precede creation)","corr of two recordings + lifecycle nulls + ordering","demote; resolution_time_hours = duration source of truth"),
 ("F03","HIGH","lifecycle violated: csat on 98% of unresolved (~59.4k invalid); notes inverted","null rates by status","csat_valid (closed only, 40,217); notes demoted"),
 ("F04","MED","customer_segment contradicts account_type (9,953 impossible combos)","crosstab + null-structure alignment","demote; account_type authoritative"),
 ("F05","MED","customer_tenure_months random (corr -0.003 with account age)","corr vs derived age","recompute tenure_months_calc"),
 ("F06","MED","category fractured into 32 spellings (20,831 rows)","value_counts two-tier structure","mapping file -> 10 categories / 5 groups; assert unmapped=0"),
 ("F07","MED","issue_description independent of category (top share ~0.10)","crosstab top-share","demote; designed taxonomy instead of text-derived"),
 ("F08","MED","resolution_notes independent of description & category; inverted lifecycle","crosstab + status nulls","demote; exclude from agent schema"),
 ("F09","MED","monthly_contract_value contradicts subscription_type (Free 67% charged)","zero-share by subscription","never use pair together; no revenue analysis"),
 ("F10","LOW","8 sentinel dates (1970/2099); scope: data Jul23-Dec24, brief says 2024-25, 2025 has 10 rows","min/max + monthly counts","drop 8; scope = reporting decision, documented"),
 ("F11","LOW","779 priority/SLA mismatches vs 99.2% conformance","priority x target crosstab","correct to canonical BEFORE breach recalc"),
 ("F12","LOW","14,750 rows first_response > resolution","row-wise ordering","flag only; values untouched"),
 ("F13","LOW","674 accounts created after their own ticket","row-wise ordering","tenure nulled + flagged"),
 ("F14","LOW","50/53 negative resolutions exactly -24h (systematic-shift signature)","distribution of violation magnitude","recoverable class if production data; demoted here with F02"),
 ("F15","LOW","43 near-duplicates (all non-key columns identical)","duplicated() excl. primary key","flag is_duplicate, keep first, never delete"),
 ("F16","LOW","previous_tickets non-monotonic random fill","per-customer monotonicity","rebuild as time-ordered cumulative count"),
 ("F17","INFO","60% of accounts show multiple names/emails; account attributes stable","per-id nunique contrast","NOT an error: ticket-level contacts; no fix"),
 ("F18","INFO","uniform distributions across status/escalation/csat: synthetic fingerprint","normalized value_counts","insights are methodological, not commercial"),
], columns=["id", "severity", "finding", "detection", "treatment"])

findings.to_csv(OUT / "dq_findings.csv", index=False)
print(f"exported -> outputs/dq_findings.csv  ({len(findings)} findings)")
findings[["id", "severity", "finding"]]

exported -> outputs/dq_findings.csv  (18 findings)


,id,severity,finding
0,F01,HIGH,sla_breached flag disconnected from handling t...
1,F02,HIGH,ticket_resolved_date decoupled from resolution...
2,F03,HIGH,lifecycle violated: csat on 98% of unresolved ...
3,F04,MED,"customer_segment contradicts account_type (9,9..."
4,F05,MED,customer_tenure_months random (corr -0.003 wit...
5,F06,MED,"category fractured into 32 spellings (20,831 r..."
6,F07,MED,issue_description independent of category (top...
7,F08,MED,resolution_notes independent of description & ...
8,F09,MED,monthly_contract_value contradicts subscriptio...
9,F10,LOW,8 sentinel dates (1970/2099); scope: data Jul2...


### Conclusion

The trustworthy skeleton: `ticket_id, customer_id, account_type, region, service_area, priority,
status, channel, operating_system, created_date` (post-sentinel). Every headline metric — SLA breach,
duration, tenure, ticket history, CSAT — required rebuilding or rescoping from internally consistent inputs.

**Method takeaway**: the audit is complete not because no more issues turn up, but because every
applicable test pattern (dual recordings, lifecycle, entity stability, row ordering) has been
exhausted against every eligible column pair. Treatment rules are implemented and verified in
`src/m2_clean.py` against the baselines above.